# Exercise 5-1-1: Detecting Non-Answers in Earnings Conference Calls

**The research task.** In the Q&A section of an earnings call, analysts ask questions and managers answer them — except when they don't. A **non-answer** is a response in which the manager signals an inability or an unwillingness to provide (part of) the information that was asked for: *"we don't disclose that", "it's too early to say", "we don't give guidance on that."* Hollander, Pronk and Roelofsen (2010, *JAR* 48(3):531–563) show these silences are informative, but they had to hand-code them, which took months. Gow, Larcker and Zakolyukina (2021, *JAR* 59(4):1349–1384) automate the coding with regular expressions; in de Kok's replication that measure reaches 86% accuracy but only a 0.49 F1 score on the non-answer class, missing 57% of the true non-answers.

Deciding whether a response is a non-answer requires reading the answer **in the context of the question**, tolerating genuine ambiguity, and coping with messy transcripts — precisely the kind of judgement task that used to require a research assistant, and precisely where a generative LLM earns its cost. de Kok (2025) uses ChatGPT prompt alone and gets 91% accuracy (0.72 F1); the full pipeline reaches 96% (0.87 F1), a 70% reduction in the error rate relative to Gow et al. (2021).

**What we build here** is a simple replication of de Kok (2025) Table 1 (column (3)): raw transcript in → one row per Q&A pair → a 0/1 non-answer label out, plus an evaluation of whether that label can be trusted.

de Kok's four-step framework (§3 of the paper) maps onto this notebook as follows:

| Framework step | Where in this notebook |
| --- | --- |
| 1. Define and understand your problem | Step 1 |
| 2. Decide on the approach and model | Step 3 |
| 3. Develop your prompt | Step 4 |
| 4. Evaluate the construct validity | Step 6 |


In [ ]:
import json
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

import os

load_dotenv()

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### Step 1. From a raw transcript to Q&A pairs

The unit of observation in de Kok (2025) is the **question–answer pair**, not the conference calls.

In [ ]:
example_file = Path("../data/conference_call_transcript/TSLA_2024Q4.txt")

raw = example_file.read_text(encoding="utf-8")
print(raw[:900])

#### 1.1 Keep only the Q&A section

In [ ]:
presentation, sep, qa = raw.partition("# Question-and-Answer Session")
assert sep, f"No Q&A section found in {example_file.name}"

print(f"presentation: {len(presentation):>6,} characters")
print(f"Q&A section : {len(qa):>6,} characters")

#### 1.2 Split the Q&A section into speaker turns

One regular expression finds every speaker header; the text of a turn is everything up to the *next* header. Two details worth noting:

* `(?:\*(?P<title>[^*\n]*)\*\n)?` makes the title line **optional** because some operator has no title.
* Some transcript vendors prefix names with `Q - ` / `A - `; we strip that.

In [ ]:
TURN_RE = re.compile(r"^\*\*(?P<speaker>[^*\n]+)\*\*\n(?:\*(?P<title>[^*\n]*)\*\n)?", re.M)

def split_turns(section: str) -> list[dict]:
    """Split a Q&A section into a list of {speaker, title, text} turns."""
    turns = []
    headers = list(TURN_RE.finditer(section))
    for i, header in enumerate(headers):
        end = headers[i + 1].start() if i + 1 < len(headers) else len(section)
        turns.append({
            "speaker": re.sub(r"^[QA] - ", "", header.group("speaker")).strip(),
            "title": (header.group("title") or "").strip(),
            # collapse the hard line breaks inside a turn into single spaces
            "text": " ".join(section[header.end():end].split()),
        })
    return turns

turns = split_turns(qa)
print(f"{len(turns)} speaker turns")
pd.DataFrame(turns).head(6)

#### 1.3 Who is asking and who is answering?

In [ ]:
MGMT_TITLE = re.compile(
    r"CEO|CFO|CTO|Chief|President|Officer|Chairman|Founder|Technoking|"
    r"Architect|Executive|Engineering|Director|VP|Tesla",
    re.I,
)
IR_TITLE = re.compile(r"Investor Relations|\bIR\b", re.I)

def role(turn: dict) -> str:
    """Classify a speaker turn as operator / ir / management / analyst."""
    if turn["speaker"].lower().startswith("operator"):
        return "operator"
    if IR_TITLE.search(turn["title"]):
        return "ir"
    if MGMT_TITLE.search(turn["title"]) or "Company Representative" in turn["speaker"]:
        return "management"
    return "analyst"

turns_df = pd.DataFrame(turns).assign(role=lambda d: d.apply(role, axis=1))
turns_df.groupby(["role", "speaker", "title"]).size().rename("turns").reset_index()

#### 1.4 Build the Q&A pairs

The pairing rule:

* a **question** is a turn by an analyst (or journalist), or an IR turn that contains a question mark;
* the **answer** is every management turn that follows it, concatenated — if the CFO adds to the CEO's reply, that is still one answer to one question.

We then apply the length filters, which drop the "Q: Thank you – A: Thanks!" pairs that would otherwise inflate the accuracy statistics: question ≥ 30 characters, answer ≥ 10 characters, and the two together ≥ 75 characters.

In [ ]:
MIN_Q, MIN_A, MIN_BOTH = 30, 10, 75  # de Kok's (2025) footnote 7

def qa_pairs(turns: list[dict]) -> list[dict]:
    """Pair each question turn with the management turns that answer it."""
    pairs, i = [], 0
    while i < len(turns):
        turn = turns[i]
        turn_role = role(turn)
        asks_question = turn_role == "analyst" or (turn_role == "ir" and "?" in turn["text"])
        if not asks_question:
            i += 1
            continue

        answer_parts, j = [], i + 1
        while j < len(turns) and role(turns[j]) == "management":
            answer_parts.append(turns[j]["text"])
            j += 1
        answer = " ".join(answer_parts)

        if (len(turn["text"]) >= MIN_Q and len(answer) >= MIN_A
                and len(turn["text"]) + len(answer) >= MIN_BOTH):
            pairs.append({
                "asker": turn["speaker"],
                "asker_role": turn_role,
                "question": turn["text"],
                "answer": answer,
            })
        i = max(j, i + 1)
    return pairs

def load_call(path: Path) -> pd.DataFrame:
    """Read one transcript file and return its Q&A pairs as a DataFrame."""
    _, _, section = path.read_text(encoding="utf-8").partition("# Question-and-Answer Session")
    df = pd.DataFrame(qa_pairs(split_turns(section)))
    df.insert(0, "call", path.stem)
    df.insert(1, "pair_id", [f"{path.stem}_{k:03d}" for k in range(len(df))])
    return df

pairs = load_call(example_file)
print(f"{len(pairs)} Q&A pairs in {example_file.stem}")
pairs.head()

### Step 2. Develop the prompt

**First decide on the approach and model**

Three ways to instruct a generative LLM (de Kok, §3.2):

| Approach | What it is | Trade-off |
| --- | --- | --- |
| **Zero shot** | instructions + data, no examples | easiest; least control |
| **Few shot** | instructions + a handful of labelled examples in every prompt | better on nuanced tasks; more tokens per observation |
| **Fine-tuning** | retrain the model weights on a labelled training set | most control; needs a training set and much more work |

Here we just use zero-shot, which corresponds to column (3) of his Table 1.

---

Four ideas from the paper are built into the prompt below (§3.3 and Appendix C):

1. **Give the model the coding rules**, including what is *not* a non-answer. Boilerplate forward-looking disclaimers followed by a real answer are the single biggest source of false positives.
2. **Make the output machine-readable.** We use a Pydantic schema with `client.responses.parse()`, so the completion comes back as a validated object instead of prose we have to regex.
3. **Ask for the assessment *before* the label.** LLMs generate left to right, so their own reasoning becomes part of the context for the token that follows — the classic chain-of-thought effect. Field order in the schema is therefore a design decision, not cosmetics.
4. **State the expected distribution.** Each call is independent, so the model has no idea that non-answers are rare and will happily flag a third of the sample. de Kok reports that deleting his "these sentences are rare, in 65% of the cases..." sentence *significantly* degrades performance.

In [ ]:
INSTRUCTIONS = """
You are a research assistant coding earnings conference call transcripts for an
academic accounting study. Apply the coding rules exactly as written and base your
coding only on the text you are given.
"""

PROMPT = """
Below is one question-and-answer pair from the Q&A section of an earnings conference call.

Analyst question:
{question}

Manager response:
{answer}

A NON-ANSWER is a response in which the manager indicates an inability or an unwillingness
to provide (part of) the information the question asked for. It includes:
- declining for proprietary, competitive, policy or legal reasons ("we don't disclose that",
  "we don't give guidance on that");
- saying they do not know, do not have the information, or that it is too early to tell;
- deflecting the question with a purely qualitative statement or a broad range instead of
  the specific information that was requested.

It is NOT a non-answer when the manager:
- provides the requested information, even briefly, roughly or imprecisely;
- adds a boilerplate disclaimer (e.g. about forward-looking statements) and then answers;
- discusses general uncertainty about the future while still answering the question asked.

Your task:
1. Assess the response in the context of the question that was asked. If it contains a
   non-answer, quote the sentence(s) from the response that show it.
2. Then classify the pair: 1 if the response contains a non-answer, 0 otherwise.

Important: non-answers are relatively rare - for roughly 85% of pairs the correct
classification is 0. It is fine, and expected, to return 0 most of the time.
"""

In [ ]:
class NonAnswerCoding(BaseModel):
    """Schema for the completion. Field order = the order the model generates them in."""

    assessment: str = Field(
        description="Two or three sentences assessing the response in the context of the question."
    )
    evidence: str = Field(
        description="Verbatim sentence(s) from the response showing the non-answer; "
                    "an empty string if there is none."
    )
    nonanswer: int = Field(
        description="1 if the response contains a non-answer, 0 otherwise."
    )

In [ ]:
def classify_pair(question: str, answer: str, model: str = 'gpt-5.5') -> tuple[NonAnswerCoding, dict]:
    """Classify one Q&A pair. Returns the parsed coding and a raw record for the log."""
    prompt = PROMPT.format(question=question, answer=answer)
    response = client.responses.parse(
        model=model,
        instructions=INSTRUCTIONS,
        input=prompt,
        text_format=NonAnswerCoding,
    )
    coding = response.output_parsed
    record = {
        "model": model,
        "prompt": prompt,
        "completion": coding.model_dump(),
        "usage": response.usage.model_dump() if response.usage else {},
    }
    return coding, record

In [ ]:
# One pair, to see what comes back before spending money on the rest.
coding, record = classify_pair(pairs["question"][0], pairs["answer"][0])
coding

In [ ]:
# Tokens drive both the cost and the speed - check them before scaling up.
print(f"input tokens: {record['usage'].get('input_tokens')}, "
      f"output tokens: {record['usage'].get('output_tokens')}")

### Step 3. Run the classification over the call

Two things the loop below does that a throwaway script would not, both from §5.2 of the paper:

* **It logs the raw prompt and the raw completion for every observation** to a JSONL file. Third-party providers retire and silently change models; generations are not perfectly reproducible even at the same model version. Your prompts and completions are the primary data of the study — treat them the way you treat a raw WRDS download, and never regenerate them casually.
* **It skips pairs that are already in the log.** A crashed run resumes where it stopped, and re-running the cell costs nothing. (Delete the log file to force a fresh run.)

In [ ]:
RAW_LOG = Path("../data/nonanswer_raw.jsonl")

def load_raw(log: Path = RAW_LOG) -> pd.DataFrame:
    """Read the raw log back into a DataFrame, one row per classified pair."""
    if not log.exists():
        return pd.DataFrame(columns=["pair_id", "call", "assessment", "evidence", "nonanswer"])
    records = [json.loads(line) for line in log.read_text().splitlines() if line.strip()]
    return pd.DataFrame([
        {"pair_id": r["pair_id"], "call": r["call"], **r["completion"]} for r in records
    ]).drop_duplicates(subset="pair_id", keep="last")

def classify_frame(df: pd.DataFrame, model: str = 'gpt-5.5',
                   log: Path = RAW_LOG, max_workers: int = 4) -> None:
    """Classify every pair in `df` that is not already in the log, appending as we go."""
    done = set(load_raw(log)["pair_id"])
    todo = df[~df["pair_id"].isin(done)]
    print(f"{len(todo)} pairs to classify, {len(df) - len(todo)} already in the log")
    if todo.empty:
        return

    with ThreadPoolExecutor(max_workers=max_workers) as pool, log.open("a") as fh:
        futures = {
            pool.submit(classify_pair, row.question, row.answer, model): row
            for row in todo.itertuples()
        }
        for n, future in enumerate(as_completed(futures), start=1):
            row = futures[future]
            try:
                _, record = future.result()
            except Exception as exc:  # keep the completed work, report the failure
                print(f"\n{row.pair_id} failed: {exc}")
                continue
            fh.write(json.dumps({"pair_id": row.pair_id, "call": row.call, **record}) + "\n")
            fh.flush()
            print(f"\r{n}/{len(futures)} classified", end="")
    print()

In [ ]:
classify_frame(pairs)

In [ ]:
results = pairs.merge(load_raw().drop(columns="call"), on="pair_id")
results

In [ ]:
print(f"{results['nonanswer'].mean():.1%} of the {len(results)} Q&A pairs in "
      f"{example_file.stem} contain a non-answer")
print("(de Kok (2025) finds 13.9% across 1.15 million pairs)")